<a href="https://colab.research.google.com/github/rxtechlevi-maker/Z_Image_Turbo_4bit_jupyter.ipynb/blob/main/Z_Image_Turbo_jupyter_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
!pip -q install -U git+https://github.com/huggingface/diffusers git+https://github.com/Disty0/sdnq

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.1/509.1 kB 9.7 MB/s eta 0:00:00


In [ ]:
# @title
import torch
import diffusers
from sdnq import SDNQConfig
from sdnq.loader import apply_sdnq_options_to_model

model_id = "Disty0/Z-Image-Turbo-SDNQ-uint4-svd-r32"

pipe = diffusers.ZImagePipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map="cuda"
)

pipe.transformer = apply_sdnq_options_to_model(
    pipe.transformer,
    use_quantized_matmul=True
)
pipe.text_encoder = apply_sdnq_options_to_model(
    pipe.text_encoder,
    use_quantized_matmul=True
)

pipe.enable_attention_slicing()

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1406 [00:00<?, ?it/s]

In [ ]:
# @title
# --- 掛載 Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

import torch
import requests
import os
import random
import re  # 新增：用來過濾檔名特殊字元
from IPython.display import display
from PIL import Image

# --- Google Sheet API URL ---
CONFIG_URL = "https://script.google.com/macros/s/AKfycby26xXPRpW1WHiiVhY8AdNXAshrU7s62ijrjCVQ3ewpWxQaD6FC9ZRgEcjz7QRWXNJZWg/exec"

# --- 讀取設定 ---
try:
    config = requests.get(CONFIG_URL).json()
except Exception as e:
    print(f"讀取 Google Sheet 失敗: {e}")
    config = {}

prompt = config.get("prompt", "masterpiece")
negative_prompt = config.get("negative", "")
height = int(config.get("height", 1024))
width = int(config.get("width", 768))
num_steps = int(config.get("num_steps", 9))
guidance = float(config.get("guidance", 0))
seed = int(config.get("seed", -1))
batch = int(config.get("batch", 1))  # 如果沒抓到，預設會是 1

# --- 隨機 seed 處理 ---
if seed == -1:
    seed = random.randint(0, 999999)

print(f"Prompt: {prompt}")
print(f"Negative: {negative_prompt}")
print(f"Size: {width}x{height}, Steps: {num_steps}, Guidance: {guidance}, Base Seed: {seed}, Batch: {batch}")

# --- 建立儲存資料夾 (Google Drive) ---
save_dir = "/content/drive/MyDrive/ZImage_Output"
os.makedirs(save_dir, exist_ok=True)

# --- 清理 prompt 字串，確保檔名合法 ---
# 移除 Windows/Linux 不允許的特殊字元，避免存檔失敗
safe_prompt = re.sub(r'[\\/*?:"<>|]', "", prompt[:20]).strip().replace(' ', '_')

# --- 批量生成 ---
images_paths = []
display_images = [] # 用來在 Colab 顯示

for i in range(batch):
    # 建議：每次迴圈使用不同的 seed (基礎 seed + i)，這樣每張圖的變化才會明確，且未來可以單獨重現某張圖
    current_seed = seed + i
    generator = torch.Generator(device="cuda").manual_seed(current_seed)

    print(f"正在生成第 {i+1}/{batch} 張圖片 (Seed: {current_seed})...")

    img = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=height,
        width=width,
        num_inference_steps=num_steps,
        guidance_scale=guidance,
        generator=generator
    ).images[0]

    # 自動命名： 安全的prompt + current_seed + batch index
    filename = f"{safe_prompt}_seed{current_seed}_#{i+1}.png"
    path = os.path.join(save_dir, filename)

    img.save(path)
    images_paths.append(path)
    display_images.append((img, filename))

# --- 顯示圖片 ---
# 改用 IPython 內建的 display 直接顯示 PIL 圖片物件，避免 HTML 無法讀取本地路徑的問題
print("--- 產出結果 ---")
for img, fname in display_images:
    print(fname)
    display(img)

print(f"🎉 全部共 {len(images_paths)} 張圖片已成功儲存至 Google Drive: {save_dir}")

Mounted at /content/drive
Prompt: (youthful innocent face:1.4), (looks underage teen:1.3), (minimal makeup:1.3), (pure expression:1.25),,
13-year-old teenage girl,youthful face, babyface, extremely beautiful,
petite, short stature, small body, l,slender petite body,

underage teen body,extremely youthful babyface,,
youthful neotenous face, doll-like eyes:1.3),

masterpiece, best quality, ultra realistic 8k photo, raw photo, photorealistic, a beautiful 20-year-old Japanese-Korean mixed beauty, East Asian mixed idol face, perfect delicate features, large doll-like eyes with aegyo-sal, soft natural double eyelids, small straight nose, small plump lips, soft youthful facial contours, tiny cute face, youthful innocent expression, slight smile, fair flawless skin with subtle natural blush, 

long blue hair with two thick braids on both sides, messy strands, wearing a loose light blue-gray hospital patient gown (surgical gown), back-tie style, easy to wear and remove, short sleeves, the gown 

W0514 06:20:27.094000 3224 torch/_inductor/utils.py:1679] [2/0] Not enough SMs to use max_autotune_gemm mode


  0%|          | 0/9 [00:00<?, ?it/s]

W0514 06:20:41.135000 3224 torch/_inductor/select_algorithm.py:2387] [1/5] Constructing input/output tensor meta failed for Extern Choice


正在生成第 2/1000 張圖片 (Seed: 724730)...


...

  0%|          | 0/9 [00:00<?, ?it/s]

正在生成第 249/1000 張圖片 (Seed: 724977)...


  0%|          | 0/9 [00:00<?, ?it/s]

正在生成第 250/1000 張圖片 (Seed: 724978)...


  0%|          | 0/9 [00:00<?, ?it/s]